# Notebook 2 — K-Means Segmentation & Feature Extraction
**Project:** Chest X-ray Pneumonia Detection — Generalization Study

This notebook:
1. Loads preprocessed image arrays from Notebook 1
2. Finds the optimal number of clusters using Elbow Method + Silhouette Score
3. Applies K-Means to segment each X-ray into regions
4. Extracts a compact feature vector from each image
5. Saves feature matrices for use in Notebook 3

## Step 0 — Install & import libraries

In [ ]:
!pip install scikit-learn numpy matplotlib tqdm --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from tqdm import tqdm
import os

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

## Step 1 — Load preprocessed arrays from Notebook 1

In [ ]:
X_train = np.load('preprocessed/X_train.npy')
y_train = np.load('preprocessed/y_train.npy')
X_test  = np.load('preprocessed/X_test.npy')
y_test  = np.load('preprocessed/y_test.npy')
X_nih   = np.load('preprocessed/X_nih.npy')
y_nih   = np.load('preprocessed/y_nih.npy')

print(f'Loaded: X_train={X_train.shape}, X_test={X_test.shape}, X_nih={X_nih.shape}')

## Step 2 — Find optimal k using Elbow Method + Silhouette Score
We test k=2 to k=6 on a small sample to keep runtime fast.

In [ ]:
# Use a small sample of 500 images for cluster selection (fast)
sample_idx = np.random.choice(len(X_train), size=500, replace=False)
sample = X_train[sample_idx].reshape(500, -1)  # flatten to pixel vectors

k_values = range(2, 7)
inertias, silhouette_scores = [], []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=5)
    labels = km.fit_predict(sample)
    inertias.append(km.inertia_)
    sil = silhouette_score(sample, labels, sample_size=200, random_state=SEED)
    silhouette_scores.append(sil)
    print(f'k={k}  inertia={km.inertia_:.0f}  silhouette={sil:.4f}')

# Plot both metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(k_values), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('SSE (Inertia)')
axes[0].set_title('Elbow Method (SSE vs. k)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_values), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs. k')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as elbow_silhouette.png')

## Step 3 — Train final K-Means model with k=3
k=3 captures three natural regions: background (dark), lung tissue (mid), abnormalities (bright).

In [ ]:
K = 3  # optimal number of clusters

# Use MiniBatchKMeans for speed on full dataset
print('Training K-Means on training images...')
kmeans = MiniBatchKMeans(n_clusters=K, random_state=SEED, n_init=5, batch_size=512)
train_flat = X_train.reshape(len(X_train), -1)
kmeans.fit(train_flat)
print(f'K-Means trained. Cluster centers shape: {kmeans.cluster_centers_.shape}')

## Step 4 — Visualize segmented X-rays
Show original vs. K-Means segmented image side by side.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('K-Means Segmentation Examples (k=3)', fontsize=14)

sample_indices = [0, 1, 2]  # show 3 examples
for row, idx in enumerate(sample_indices):
    img = X_train[idx]
    img_flat = img.reshape(1, -1)
    
    # Predict cluster for each pixel
    pixel_clusters = kmeans.predict(img.reshape(-1, 1) if img.ndim == 2 
                                    else img.reshape(-1, img.shape[-1]))
    # For grayscale: reshape pixel values as 1D features
    pixel_labels = kmeans.predict(img.reshape(-1, 1) 
                                  if K == 3 else img.flatten().reshape(-1,1))
    segmented = pixel_labels.reshape(img.shape)
    
    # Original
    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title(f'Original — {"Normal" if y_train[idx]==0 else "Pneumonia"}')
    axes[row, 0].axis('off')
    
    # Segmented
    axes[row, 1].imshow(segmented, cmap='viridis')
    axes[row, 1].set_title('K-Means Segmented (k=3)')
    axes[row, 1].axis('off')
    
    # Overlay
    axes[row, 2].imshow(img, cmap='gray', alpha=0.6)
    axes[row, 2].imshow(segmented, cmap='jet', alpha=0.4)
    axes[row, 2].set_title('Overlay')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('segmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as segmentation_examples.png')

## Step 5 — Extract feature vectors from all images
For each image, we extract:
- Proportion of pixels in each cluster (3 values)
- Mean pixel intensity per cluster (3 values)
- Std of pixel intensity per cluster (3 values)

Total: 9 features per image — a compact, meaningful representation.

In [ ]:
def extract_kmeans_features(images, kmeans_model, k):
    """
    For each image, compute cluster proportions, mean intensities,
    and std intensities per cluster.
    Returns feature matrix of shape (N, k*3).
    """
    features = []
    for img in tqdm(images, desc='Extracting features'):
        pixels = img.flatten()
        # Predict cluster for each pixel (treat each pixel value as 1D feature)
        pixel_labels = kmeans_model.predict(pixels.reshape(-1, 1))
        
        proportions = []
        means = []
        stds = []
        for c in range(k):
            mask = pixel_labels == c
            prop = mask.sum() / len(pixels)
            mean = pixels[mask].mean() if mask.sum() > 0 else 0.0
            std  = pixels[mask].std()  if mask.sum() > 0 else 0.0
            proportions.append(prop)
            means.append(mean)
            stds.append(std)
        
        features.append(proportions + means + stds)
    
    return np.array(features, dtype=np.float32)

# Re-fit K-Means on pixel values only (1D) for per-pixel prediction
print('Re-fitting K-Means on pixel intensities (1D) for feature extraction...')
all_pixels = X_train.flatten().reshape(-1, 1)
kmeans_1d = KMeans(n_clusters=K, random_state=SEED, n_init=5)
# Sample for speed
sample_pixels = all_pixels[np.random.choice(len(all_pixels), size=50000, replace=False)]
kmeans_1d.fit(sample_pixels)
print(f'Cluster centers (pixel intensity): {sorted(kmeans_1d.cluster_centers_.flatten())}')

In [ ]:
print('Extracting features from training set...')
X_train_feat = extract_kmeans_features(X_train, kmeans_1d, K)
print('Extracting features from test set...')
X_test_feat  = extract_kmeans_features(X_test,  kmeans_1d, K)
print('Extracting features from NIH set...')
X_nih_feat   = extract_kmeans_features(X_nih,   kmeans_1d, K)

print(f'\nFeature matrix shapes:')
print(f'  X_train_feat : {X_train_feat.shape}  (9 features per image)')
print(f'  X_test_feat  : {X_test_feat.shape}')
print(f'  X_nih_feat   : {X_nih_feat.shape}')

## Step 6 — Also create raw pixel baseline features (flattened + PCA reduced)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Flatten images to pixel vectors
X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)
X_nih_flat   = X_nih.reshape(len(X_nih), -1)

# Reduce with PCA to 50 components (keeps things manageable)
print('Applying PCA to raw pixel features (baseline)...')
scaler_raw = StandardScaler()
X_train_scaled = scaler_raw.fit_transform(X_train_flat)
X_test_scaled  = scaler_raw.transform(X_test_flat)
X_nih_scaled   = scaler_raw.transform(X_nih_flat)

pca = PCA(n_components=50, random_state=SEED)
X_train_raw = pca.fit_transform(X_train_scaled)
X_test_raw  = pca.transform(X_test_scaled)
X_nih_raw   = pca.transform(X_nih_scaled)

print(f'Raw PCA features: train={X_train_raw.shape}, test={X_test_raw.shape}, nih={X_nih_raw.shape}')
print(f'Variance explained by 50 PCs: {pca.explained_variance_ratio_.sum():.2%}')

## Step 7 — PCA visualization of K-Means features (colored by label)

In [ ]:
from sklearn.decomposition import PCA as PCA2D

pca_2d = PCA2D(n_components=2, random_state=SEED)
X_train_2d = pca_2d.fit_transform(X_train_feat)

fig, ax = plt.subplots(figsize=(8, 6))
colors = {0: '#2196F3', 1: '#F44336'}
labels_map = {0: 'Normal', 1: 'Pneumonia'}

for label in [0, 1]:
    mask = y_train == label
    ax.scatter(X_train_2d[mask, 0], X_train_2d[mask, 1],
               c=colors[label], label=labels_map[label],
               alpha=0.4, s=10)

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('PCA 2D Visualization of K-Means Features')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as pca_visualization.png')

## Step 8 — Save all feature matrices

In [ ]:
# K-Means features
np.save('preprocessed/X_train_feat.npy', X_train_feat)
np.save('preprocessed/X_test_feat.npy',  X_test_feat)
np.save('preprocessed/X_nih_feat.npy',   X_nih_feat)

# Raw PCA baseline features
np.save('preprocessed/X_train_raw.npy', X_train_raw)
np.save('preprocessed/X_test_raw.npy',  X_test_raw)
np.save('preprocessed/X_nih_raw.npy',   X_nih_raw)

print('All feature matrices saved to /preprocessed/')
print('Notebook 2 complete. Open notebook 3 (classification) next.')